# The Topics Seasonality Model — reproduce in one run
**ArtaQuest Research** · [read the article](https://artaquest.org/research/?article=seasonality)

Does the timing of what the world Googles keep time with the sky? For one topic we fit 22 years of **weekly** Google-Trends interest to **twelve sidereal cycles** and read off its best-fitting zodiac sign — a transparent least-squares curiosity. **Runtime → Run all**, then change `KEY` to try any topic.

_Correlation is not causation._

### 1 · Load the open data
The weekly search series and the pre-computed sidereal longitudes are fetched over the web — no local files, no setup.

In [ ]:
import numpy as np, pandas as pd, json, urllib.request, matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
B = 'https://artaquest.org/wp-content/uploads/research/'
ser = json.load(urllib.request.urlopen(B+'series.json'))      # 87 weekly search-interest series
eph = pd.read_csv(B+'ephemeris_weekly.csv')                   # sidereal body longitudes, weekly
print('topics available:', len(ser['series']))
print('example keys:', list(ser['topics'].keys())[:8])

### 2 · The model
Twelve bodies, each a clock at its own orbital period. We slide one **angle** around the zodiac; at each angle every body adds a localised `sinc` bump, and we fit by least squares. The angle of lowest mean-squared error is the topic's **sign** (identically, the angle of highest R²). The most recent year is cropped (it's the least reliable — see the Recency Bias notebook).

In [ ]:
BODIES=['sun','moon','mercury','venus','mars','jupiter','saturn','uranus','neptune','pluto','chiron','node']
T={'sun':1.0,'moon':0.0748,'mercury':0.241,'venus':0.615,'mars':1.881,'jupiter':11.86,'saturn':29.46,
   'uranus':84.0,'neptune':164.8,'pluto':248.0,'chiron':50.0,'node':18.6}
SIGNS=['Aries','Taurus','Gemini','Cancer','Leo','Virgo','Libra','Scorpio','Sagittarius','Capricorn','Aquarius','Pisces']
REFS=np.arange(0,360,5); DROP=52
def design(r,n): return np.column_stack([np.sinc(np.deg2rad((eph[b].values[:n]-r+180)%360-180)/T[b]) for b in BODIES])

KEY='tourism'                          # <-- change to any key from ser['topics']
y=np.array(ser['series'][KEY])[:-DROP]; n=len(y)
mse=[float(((y-LinearRegression().fit(design(r,n),y).predict(design(r,n)))**2).mean()) for r in REFS]
ref=REFS[int(np.argmin(mse))]
m=LinearRegression().fit(design(ref,n),y); pred=m.predict(design(ref,n))
r2=1-((y-pred)**2).sum()/((y-y.mean())**2).sum()
print(f"{ser['topics'][KEY]}  ->  {SIGNS[int(ref//30)%12]}   R2 = {r2*100:.1f}%   (angle {ref} deg)")

### 3 · One figure
**Left** — fit quality at every angle of the zodiac; the gold band is the winning sign. **Right** — what the world searched (blue) vs the rhythm the model predicts (gold).

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(13,3.4)); win=int(ref//30)
tv=[100*(max(mse)-v) for v in mse]
ax[0].axvspan(win*30,win*30+30,color='#E8B923',alpha=.15); ax[0].plot(REFS,tv,color='#2352E8',lw=2)
ax[0].set_title('tuning: fit quality vs zodiac angle'); ax[0].set_xlabel('angle (deg)'); ax[0].set_xticks(range(0,361,90))
ax[1].plot(y,color='#2352E8',lw=.6,alpha=.6,label='searched'); ax[1].plot(pred,color='#E8B923',lw=2,label='sky predicts')
ax[1].set_title(f"{ser['topics'][KEY]}  -  {SIGNS[int(ref//30)%12]}  (R2 {r2*100:.0f}%)"); ax[1].legend(); ax[1].set_xlabel('week')
plt.tight_layout(); plt.show()